# ML-08 · Model vs baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** a method chosen from the menu and explained, compared with the Week-4 baseline on the same
data, split and metric, with features and errors interpreted, ending with a model-vs-baseline table. **The principle it
serves:** a complex model must earn its place over a frozen baseline under identical conditions. Complexity alone earns
nothing.

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


## 1. Method choice

- **Regularised logistic regression.** Readable coefficients, stable on small training sets, and every coefficient is
  checkable against ML-06.
- **Histogram gradient boosting.** Captures interactions and handles missing values natively. Missingness here is
  structural (no GA4, no matured history yet), so that matters.
- **Rejected:** random forest (unstable across months in my archived Lane 2 work, see `work/archive/`) and deep
  models (about 50,000 rows, so no case for them).

**Tuning, nested and honest.** For each forward month *o*, tuning uses only training months (origins <= *o-2*), with an
inner GroupKFold by client scored by PR-AUC. The grid is five values of C for the logistic model and 16 random
configurations for boosting. The test month is never seen during tuning.

## 2. Features (34, engineered from the ML-06 findings)

In [2]:
feats = receipt("erosion_features.json")
fam = {"l_imp_f": "exposure", "l_clk_f": "exposure", "imp_growth": "trajectory", "ctr_growth": "trajectory", "imp_growth_prev": "trajectory",
       "imp_l14_share": "trajectory", "ctr_recency": "trajectory", "click_day_share": "stability", "spikiness": "stability",
       "pos_f": "rank", "pos_delta": "rank", "pos_std": "rank", "ctr_vs_ref": "CTR vs expectation", "ctr_vs_client": "CTR vs expectation"}
t = pd.DataFrame([{"feature": k, "family": fam.get(k, "client context" if k.startswith(("c_", "client")) else "page history" if k.startswith("page_")
                   else "channels (GA4)" if k in ("has_ga4", "l_org_sess", "has_ai_referral") else "content"), "meaning": v} for k, v in feats.items()])
display(t)

,feature,family,meaning
0,l_imp_f,exposure,"log impressions, feature window (2 months)"
1,l_clk_f,exposure,"log clicks, feature window"
2,imp_growth,trajectory,log impressions month -1 vs month -2
3,ctr_growth,trajectory,log CTR month -1 vs month -2 (clicks growth mi...
4,imp_growth_prev,trajectory,log impressions month -2 vs month -3
5,imp_l14_share,trajectory,share of month -1 impressions in its last 14 days
6,ctr_recency,trajectory,log CTR last 14 days vs whole feature window
7,click_day_share,stability,days with a click / days with an impression
8,spikiness,stability,max single-day impressions / window impressions
9,pos_f,rank,"impression-weighted position, feature window"


Every feature is computed from months *o-3* to *o-1*, or from labels whose outcome window closed by *o-1*. The list is
exported from code (`erosion_features.json`), so the paper, this table and the model use the same list.

## 3. Forward results: three development months

In [3]:
if DATA and not os.path.exists(os.path.join(OUT, "erosion_model_metrics.json")):
    import erosion_model; erosion_model.run(erosion_model.DEV_ORIGINS, "")
dev = receipt("erosion_model_metrics.json")["origins"]
names = ["logreg", "hgb", "rule_position_slipping", "mean_reversion_ctr_excess", "client_shock", "persistence_prev_ratio"]
rows = []
for o, v in dev.items():
    for n in names:
        m = v["metrics"][n]
        rows.append({"origin": o, "ranker": n, "base_rate": v["base_rate"], "p@50": m["p@50"], "p@50_ci95": v["bootstrap"][n]["p@50_ci"],
                     "pr_auc": m["ap"], "auc": round(m["auc"], 3), "within_client_p@10": round(m["within_client_p@10"], 3)})
display(pd.DataFrame(rows))
print({o: (v["best_baseline"], v["n_test"], v["test_clients"]) for o, v in dev.items()})

,origin,ranker,base_rate,p@50,p@50_ci95,pr_auc,auc,within_client_p@10
0,2026-02,logreg,0.2273,0.52,"[0.4, 0.68]",0.3166,0.602,0.487
1,2026-02,hgb,0.2273,0.44,"[0.3, 0.52]",0.3170,0.634,0.463
2,2026-02,rule_position_slipping,0.2273,0.46,"[0.26, 0.56]",0.1985,0.393,0.338
3,2026-02,mean_reversion_ctr_excess,0.2273,0.26,"[0.06, 0.42]",0.2335,0.519,0.237
4,2026-02,client_shock,0.2273,0.34,"[0.12, 0.38]",0.2101,0.439,0.188
5,2026-02,persistence_prev_ratio,0.2273,0.28,"[0.12, 0.5]",0.2060,0.469,0.212
6,2026-03,logreg,0.3382,0.60,"[0.32, 0.84]",0.4507,0.623,0.656
7,2026-03,hgb,0.3382,0.74,"[0.28, 0.82]",0.4480,0.634,0.533
8,2026-03,rule_position_slipping,0.3382,0.76,"[0.36, 0.86]",0.3851,0.500,0.400
9,2026-03,mean_reversion_ctr_excess,0.3382,0.38,"[0.16, 0.6]",0.3171,0.464,0.311


{'2026-02': ('rule_position_slipping', 7083, 20), '2026-03': ('rule_position_slipping', 8613, 27), '2026-04': ('client_shock', 11186, 28)}


**Reading it.** The base rate is 0.23 to 0.34.
- **Logistic model (P@50):** 0.52, 0.60 and 0.66, about twice the base rate every month.
- **"Top position + slipping" rule (P@50):** 0.46, 0.76 and 0.50. It wins March and loses February and April.
- **Best baseline:** a different one every month (the rule, the rule, then client shock).
- **Whole ranking (PR-AUC):** the logistic model beats every baseline in all three months (0.317, 0.451, 0.378).

The bootstrap intervals (resampling clients) are wide at 50 pages, so single-month P@50 differences are not decisive.
Consistency across months is the signal.

## 4. What the model learned, and whether it makes sense

In [4]:
sealed = receipt("erosion_model_metrics_sealed.json")
print("logistic coefficients (standardised), model trained for the sealed month:"); display(pd.DataFrame(sealed["logreg_coef_last_origin"]))
print("permutation importance (PR-AUC drop), gradient boosting, sealed month:"); display(pd.Series(sealed["permutation_importance_last_origin"]).to_frame("pr_auc_drop"))

logistic coefficients (standardised), model trained for the sealed month:


,most_negative,most_positive
l_clk_f,-2.267,NaN
click_day_share,-0.462,NaN
intent_transactional,-0.312,NaN
intent_informational,-0.283,NaN
intent_commercial,-0.239,NaN
ctr_recency,-0.216,NaN
c_l_pages,NaN,0.166
imp_growth,NaN,0.171
ctr_vs_client,NaN,0.203
ctr_growth,NaN,1.082


permutation importance (PR-AUC drop), gradient boosting, sealed month:


,pr_auc_drop
ctr_growth,0.01973
l_age,0.01960
spikiness,0.01689
page_prev_ratio,0.01661
c_l_pages,0.00698
imp_l14_share,0.00619
imp_growth,0.00603
l_imp_f,0.00543
ctr_vs_client,0.00375
ctr_vs_ref,0.00311


**Interpretation.** The largest positive weights are `l_imp_f` with a matching negative `l_clk_f` (together: many
impressions per click), `ctr_vs_ref` (the page earns more clicks than its position normally does) and `ctr_growth` (a
recent CTR bump). Read together: **the pages that lose clicks are the ones currently earning more than their rank
normally earns, or riding a recent bump.** That is the opposite of the editor's rule ("slipping keeps slipping") and it
agrees with ML-07 check 2.

**Is this just measurement noise reverting?** No. The one-signal mean-reversion rule (CTR above peers) ranks at about
the base rate (PR-AUC 0.223 to 0.317), and the label already shrinks noisy CTRs. The model's edge comes from combining
the signals, not from any one of them.

## 5. Which model to freeze, and why

In [5]:
fz = receipt("erosion_freeze.json")
print(fz["final_model"], "|", fz["reason"]); print("post-freeze change:", fz["post_freeze_changes"][0]["verification"])

logreg | Chosen on dev origins only: beats the best baseline on PR-AUC in 3/3 dev origins and on P@50 in 2/3; HGB is unstable when trained on one origin (Feb P@50 0.32). Nested rule+model blend selected weight 1.0 (pure model) in 3/3 origins, so no blend.
post-freeze change: after the change every logreg and baseline metric on the 3 dev origins and the sealed origin is identical to the frozen run; only hgb differs (hgb is not bit-reproducible across runs on this machine, another reason it was not frozen)


**Decision.** I froze the logistic model before opening the sealed month:
- it beats the best baseline on PR-AUC in 3 of 3 development months and on P@50 in 2 of 3;
- gradient boosting was unstable when trained on a single month (February);
- gradient boosting also turned out not to be bit-reproducible across reruns on this machine. Its scores moved
  between runs with identical parameters, while logistic regression and every baseline reproduced exactly.

The nested rule-plus-model blend chose a model weight of 1.0 in every month, so no blend is used.

**Said plainly:** gradient boosting scored higher than the logistic model at 50 pages in some months of the run above
(see the table in section 3 and the sealed receipt). Those numbers are reported, not claimed: a score that changes
between identical reruns cannot be frozen, and choosing the model with the best test-month score would be tuning on the
test.

## 6. Error notes

Detailed failure examples are in ML-09. In short, the top of the queue is dominated by pages with large
month-over-month impression swings. Some of these are real erosions, others are demand or tracking events, and the
human-review step in ML-10 exists for exactly those.

## 7. Model vs baseline (development months)

In [6]:
tab = pd.DataFrame({o: {"base rate": v["base_rate"], "rule (frozen) P@50": v["metrics"]["rule_position_slipping"]["p@50"],
                       "best baseline P@50": max(v["metrics"][b]["p@50"] for b in names[2:]), "logistic P@50": v["metrics"]["logreg"]["p@50"],
                       "best baseline PR-AUC": max(v["metrics"][b]["ap"] for b in names[2:]), "logistic PR-AUC": v["metrics"]["logreg"]["ap"]}
                    for o, v in dev.items()}).T
tab.loc["mean"] = tab.mean().round(3)
display(tab)

,base rate,rule (frozen) P@50,best baseline P@50,logistic P@50,best baseline PR-AUC,logistic PR-AUC
2026-02,0.2273,0.460,0.460,0.520,0.2335,0.3166
2026-03,0.3382,0.760,0.760,0.600,0.3851,0.4507
2026-04,0.3079,0.500,0.560,0.660,0.3707,0.3777
mean,0.2910,0.573,0.593,0.593,0.3300,0.3820
